In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import  FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()

text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)

In [6]:
embeddings=OllamaEmbeddings(
    model="embeddinggemma"
)
db=FAISS.from_documents(docs,embeddings)
db

In [7]:
###querying
query="How does the speaker describe the desired outcome of the war ? "
docs=db.similarity_search(query)
docs


[Document(id='19debf06-5d72-48ac-8028-d4e2a19d1b1b', metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(id='6d7d0f56-7e04-46ca-a6a4-c0faa2aa7fe9', metadata={'sour

### We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods,which largely work with retrievers.

In [8]:
retriever=db.as_retriever()
retriever.invoke(query)

[Document(id='19debf06-5d72-48ac-8028-d4e2a19d1b1b', metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(id='6d7d0f56-7e04-46ca-a6a4-c0faa2aa7fe9', metadata={'sour

### Similarity search with score

In [9]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='19debf06-5d72-48ac-8028-d4e2a19d1b1b', metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
  np.float32(0.9575441)),
 (Document(id='6d7d0f56-7e04-46ca-a6a4-c0

In [10]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-0.07503,
 0.09547105,
 0.01169296,
 -0.004354711,
 -0.014139728,
 0.08756628,
 -0.050345708,
 0.005425702,
 0.0011130997,
 -0.056803223,
 0.101846874,
 0.0014716513,
 0.011629287,
 0.023744535,
 0.019315865,
 -0.013650958,
 0.004311898,
 0.0015899366,
 -0.037788518,
 -0.0056193555,
 -0.036518782,
 -0.0649919,
 -0.003740086,
 0.0059963227,
 -0.031419802,
 0.06529511,
 0.06455855,
 -0.016996393,
 -0.024727467,
 0.03289734,
 -0.001124014,
 0.016097765,
 0.07647447,
 -0.005637967,
 -0.04624087,
 -0.024588566,
 -0.043386444,
 -0.0260209,
 -0.033465277,
 -0.015979772,
 -0.030023744,
 0.047474638,
 -0.03235683,
 -0.022787815,
 0.011749727,
 0.0076136086,
 -0.036511816,
 0.037232697,
 -0.03330873,
 0.020850547,
 -0.039985515,
 -0.025887717,
 -0.017824313,
 -0.018687027,
 -0.04209823,
 -0.01395059,
 0.033392288,
 0.064688824,
 0.009626626,
 -0.051277004,
 -0.04158795,
 -0.0010803253,
 -0.002626636,
 0.021984274,
 0.015555403,
 0.10812873,
 0.004140351,
 0.0023364115,
 0.023103183,
 0.15117916

In [11]:
docs_and_score=db.similarity_search_by_vector(embedding_vector)

In [12]:
### saving and loading 
db.save_local("faiss_index") 

In [13]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)